In [ ]:
# Import required modules
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [ ]:
# Define the structure of data (state) that flows through the graph
class BatsmanState(TypedDict):
    runs: int                 # Total runs scored
    balls: int                # Total balls faced
    fours: int                # Number of 4s hit
    sixes: int                # Number of 6s hit

    sr: float                 # Strike Rate
    bpb: float                # Balls per boundary
    boundary_percent: float   # % of runs from boundaries
    summary: str              # Final summary text

In [ ]:
# Function to calculate Strike Rate
def calculate_sr(state: BatsmanState):
    # Formula: (runs / balls) * 100
    sr = (state['runs'] / state['balls']) * 100
    
    # Return updated value as dictionary
    return {'sr': sr}

In [ ]:
# Function to calculate Balls Per Boundary
def calculate_bpb(state: BatsmanState):
    # Total boundaries = fours + sixes
    # Formula: balls / total boundaries
    bpb = state['balls'] / (state['fours'] + state['sixes'])
    
    return {'bpb': bpb}

In [ ]:
# Function to calculate % of runs from boundaries
def calculate_boundary_percent(state: BatsmanState):
    # Runs from boundaries = (4*fours + 6*sixes)
    # Formula: (boundary runs / total runs) * 100
    boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6)) / state['runs']) * 100
    
    return {'boundary_percent': boundary_percent}

In [ ]:
# Function to generate final summary (No LLM used)
def summary(state: BatsmanState):
    # Create a readable summary string using calculated values
    summary = f"""
Strike Rate - {state['sr']}
Balls per boundary - {state['bpb']}
Boundary percent - {state['boundary_percent']}
"""
    
    return {'summary': summary}

In [18]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

# Edges
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [19]:
# Initial input data
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

# Execute the workflow
result = workflow.invoke(initial_state)

# Print final output
print(result)

{'runs': 100, 'balls': 50, 'fours': 6, 'sixes': 4, 'sr': 200.0, 'bpb': 5.0, 'boundary_percent': 48.0, 'summary': '\nStrike Rate - 200.0\nBalls per boundary - 5.0\nBoundary percent - 48.0\n'}
